<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 지식 증류 · 모델 경량화 · 06. Integer-only Quantization

## Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference

- **원 논문:** [Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference](https://openaccess.thecvf.com/content_cvpr_2018/html/Jacob_Quantization_and_Training_CVPR_2018_paper.html)
- **저자 / 발표:** Benoit Jacob et al. · CVPR 2018 (2018)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** PyTorch tensor와 autograd, softmax·cross entropy, 선형대수 기초
- **로컬 학습 데이터:** `data/field_curriculum/compression_bench.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** affine quantization, integer matrix multiplication, fake-quantization 학습을 구현한다.

**축소하거나 생략한 부분:** 원 논문은 MobileNet/COCO와 실제 ARM integer kernel을 평가했다. 실습은 작은 linear layer의 수치 오차와 QAT proxy다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2.1, Eq. (1): r=S(q-Z) | Task 1 | scale/zero point와 round-trip |
| §2.2, Eq. (2)–(4): quantized matmul | Task 2 | integer accumulator |
| §2.4: zero-point and range | Task 1–2 | train-only calibration scale 고정과 uint8 clamp |
| §3: training with simulated quantization | Task 3 | fake-quant loss/accuracy |

## 미니 재현
실수 r을 scale S와 zero-point Z로 정수 q에 대응시키고, 추론의 핵심 누산을 정수로 수행합니다.

## 포트폴리오 해설: 정확도와 효율의 계약

$$r=S(q-Z),\quad q=\operatorname{clamp}(\operatorname{round}(r/S)+Z,q_{min},q_{max})$$

- **기호 정의:** r은 실수, q는 정수, S는 scale, Z는 zero-point입니다.
- **수식의 역할:** affine mapping으로 0을 정확히 표현하고 integer accumulator 뒤 scale을 복원합니다.
- **구현 이유:** 정확도만 남기면 경량화의 비용 절감 주장을 검증할 수 없으므로,
  핵심 변환과 품질·비용 측정을 독립 함수와 method로 분리했습니다.
- **Task/코드 대응:** Task P의 논문 전용 class와 `forward`, `compute_loss`,
  `training_step`, `fit_portfolio_model`, `evaluate_portfolio_model`이
  §2.1, Eq. (1): r=S(q-Z)의 흐름을 명시합니다.
- **입출력 shape:** float activation [B, 32] → uint8/int32 → quantized logit [B, 4]
- **평가:** round-trip MSE, integer matmul 상대오차, QAT accuracy
- **원문 대비 한계:** 원 논문은 MobileNet/COCO와 실제 ARM integer kernel을 평가했다. 실습은 작은 linear layer의 수치
오차와 QAT proxy다.

경량화 실험에서는 정확도만 보지 않고 parameter·sparsity·bit·latency 중 논문이
주장한 비용을 함께 기록해야 합니다. 아래 코드는 단계별 중간값을 남깁니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** float tensor [B,D] ↔ uint8/int32 tensor [B,D]와 dequantized [B,D]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import math
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
torch.set_default_device(DEVICE)
print(ACCELERATOR.summary())


def locate(relative):
    for base in (Path.cwd(), *Path.cwd().parents):
        path = base / relative
        if path.exists():
            return path
    raise FileNotFoundError(relative)


dataset_path = locate("data/field_curriculum/compression_bench.npz")
raw = np.load(dataset_path, allow_pickle=False)
features = torch.from_numpy(raw["features"]).float().to(DEVICE)
labels = torch.from_numpy(raw["labels"]).long().to(DEVICE)
teacher_logits = torch.from_numpy(raw["teacher_logits"]).float().to(DEVICE)
teacher_weights = torch.from_numpy(raw["teacher_weights"]).float().to(DEVICE)
calibration = torch.from_numpy(raw["calibration_activations"]).float().to(DEVICE)
dense_weights = torch.from_numpy(raw["dense_weights"]).float().to(DEVICE)
train_idx = torch.from_numpy(raw["train_idx"]).long().to(DEVICE)
test_idx = torch.from_numpy(raw["test_idx"]).long().to(DEVICE)
images = torch.from_numpy(raw["images"]).float().to(DEVICE)
image_labels = torch.from_numpy(raw["image_labels"]).long().to(DEVICE)
image_train_idx = torch.from_numpy(raw["image_train_idx"]).long().to(DEVICE)
image_test_idx = torch.from_numpy(raw["image_test_idx"]).long().to(DEVICE)
token_embeddings = torch.from_numpy(raw["token_embeddings"]).float().to(DEVICE)
teacher_hidden = torch.from_numpy(raw["teacher_hidden"]).float().to(DEVICE)
teacher_attention = torch.from_numpy(raw["teacher_attention"]).float().to(DEVICE)
candidate_configs = torch.from_numpy(raw["candidate_configs"]).float()
assert features.shape == (512, 32) and teacher_logits.shape == (512, 4)
assert images.shape == (320, 1, 16, 16)
assert teacher_hidden.shape == (256, 6, 24)
print(
    dataset_path.name, "features", tuple(features.shape), "images", tuple(images.shape)
)

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2.1, Eq. (1): r=S(q-Z) / §2.4: zero-point and range
- **이 셀의 책임:** Task 1 / Task 1–2
- **Tensor shape 계약:** float tensor [B,D] ↔ uint8/int32 tensor [B,D]와 dequantized [B,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** scale/zero point와 round-trip / train-only calibration scale 고정과 uint8 clamp. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def affine_quantize(x, bits=8):
    qmin, qmax = 0, 2**bits - 1
    scale = (x.max() - x.min()).clamp_min(1e-8) / (qmax - qmin)
    zero = torch.round(torch.tensor(qmin, device=x.device) - x.min() / scale).clamp(
        qmin, qmax
    )
    q = torch.round(x / scale + zero).clamp(qmin, qmax).to(torch.int32)
    return q, scale, zero


def dequantize(q, scale, zero):
    return scale * (q.float() - zero)


q, s, z = affine_quantize(calibration)
roundtrip = dequantize(q, s, z)
assert q.min() >= 0 and q.max() <= 255 and F.mse_loss(roundtrip, calibration) < 2e-4

### 구현 단계 3 · 논문 주장 확인 실험

- **원문 위치:** §2.2, Eq. (2)–(4): quantized matmul
- **이 셀의 책임:** Task 2
- **Tensor shape 계약:** float tensor [B,D] ↔ uint8/int32 tensor [B,D]와 dequantized [B,D]
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** integer accumulator. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# PyTorch CUDA에는 int32 GEMM kernel이 없으므로 정수 누산 검증만 CPU에서 수행합니다.
x = calibration[:16].cpu()
weight = teacher_weights.cpu()
qx, sx, zx = affine_quantize(x)
qw, sw, zw = affine_quantize(weight)
integer_acc = (qx - zx.to(torch.int32)) @ (qw - zw.to(torch.int32))
quantized_output = integer_acc.float() * (sx * sw)
float_output = x @ weight
relative_error = (quantized_output - float_output).norm() / float_output.norm()
assert integer_acc.dtype == torch.int32 and relative_error < 0.03
print(f"integer matmul relative error={relative_error:.3%} (CPU int32 backend)")

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §3: training with simulated quantization
- **이 셀의 책임:** Task 3
- **Tensor shape 계약:** float tensor [B,D] ↔ uint8/int32 tensor [B,D]와 dequantized [B,D]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** fake-quant loss/accuracy. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fake_quant(x, bits=8):
    q, s, z = affine_quantize(x.detach(), bits)
    rounded = dequantize(q, s, z)
    return x + (rounded - x).detach()


model = nn.Linear(32, 4)
optimizer = torch.optim.Adam(model.parameters(), lr=0.035)
losses = []
for _ in range(50):
    optimizer.zero_grad()
    logits = F.linear(
        fake_quant(features[train_idx]), fake_quant(model.weight), model.bias
    )
    loss = F.cross_entropy(logits, labels[train_idx])
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach()))
with torch.no_grad():
    accuracy = (
        (
            F.linear(
                fake_quant(features[test_idx]), fake_quant(model.weight), model.bias
            ).argmax(-1)
            == labels[test_idx]
        )
        .float()
        .mean()
        .item()
    )
assert losses[-1] < losses[0] and accuracy > 0.85
plt.plot(losses)
plt.title(f"fake-quant training: acc={accuracy:.1%}")
plt.show()

### Task P1 · 압축 대상과 핵심 모델

tensor shape와 비용 단위를 먼저 확인한 뒤 논문 전용 class의 `forward`와
`compute_loss`에 정확도·표현 보존 계약을 명시합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §2.1, Eq. (1): r=S(q-Z) / §2.2, Eq. (2)–(4): quantized matmul / §2.4: zero-point and range / §3: training with simulated quantization
- **이 셀의 책임:** Task 1 / Task 2 / Task 1–2 / Task 3
- **Tensor shape 계약:** float tensor [B,D] ↔ uint8/int32 tensor [B,D]와 dequantized [B,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** scale/zero point와 round-trip / integer accumulator / train-only calibration scale 고정과 uint8 clamp / fake-quant loss/accuracy. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    bits: int = 8
    learning_rate: float = 0.025
    steps: int = 16


def prepare_batch() -> tuple[Tensor, Tensor]:
    return features[train_idx], labels[train_idx]


class AffineQuantizer(nn.Module):
    def __init__(self, bits: int):
        super().__init__()
        self.minimum = 0
        self.maximum = 2**bits - 1
        self.register_buffer("frozen_scale", torch.tensor(1.0))
        self.register_buffer(
            "frozen_zero_point",
            torch.tensor(0, dtype=torch.int32),
        )
        self.register_buffer("calibrated", torch.tensor(False))

    def parameters_for(self, values: Tensor) -> tuple[Tensor, Tensor]:
        scale = (values.max() - values.min()).clamp_min(1e-8)
        scale = scale / (self.maximum - self.minimum)
        zero = torch.round(self.minimum - values.min() / scale)
        return scale, zero.clamp(self.minimum, self.maximum)

    def calibrate(self, values: Tensor) -> None:
        scale, zero = self.parameters_for(values.detach())
        self.frozen_scale.copy_(scale)
        self.frozen_zero_point.copy_(zero.to(torch.int32))
        self.calibrated.fill_(True)

    def forward(self, values: Tensor, use_frozen: bool = False) -> Tensor:
        integer, scale, zero = self.quantize_to_int(values, use_frozen)
        restored = scale * (integer - zero)
        return values + (restored.float() - values).detach()

    def quantize_to_int(
        self,
        values: Tensor,
        use_frozen: bool = False,
    ) -> tuple[Tensor, Tensor, Tensor]:
        if use_frozen:
            if not bool(self.calibrated):
                raise RuntimeError("calibrate activation quantizer before inference")
            scale = self.frozen_scale
            zero = self.frozen_zero_point
        else:
            scale, zero = self.parameters_for(values.detach())
        integer = torch.round(values.detach() / scale + zero)
        integer = integer.clamp(self.minimum, self.maximum).to(torch.int32)
        return integer, scale, zero.to(torch.int32)


class IntegerQuantizationPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.activation_quantizer = AffineQuantizer(config.bits)
        self.weight_quantizer = AffineQuantizer(config.bits)
        self.classifier = nn.Linear(features.shape[1], teacher_logits.shape[1])

    def calibrate_activations(self, inputs: Tensor) -> None:
        self.activation_quantizer.calibrate(inputs)

    def forward(self, inputs: Tensor) -> Tensor:
        quantized_input = self.activation_quantizer(inputs, use_frozen=True)
        quantized_weight = self.weight_quantizer(self.classifier.weight)
        return F.linear(quantized_input, quantized_weight, self.classifier.bias)

    def integer_forward(self, inputs: Tensor) -> tuple[Tensor, Tensor]:
        input_integer, input_scale, input_zero = (
            self.activation_quantizer.quantize_to_int(
                inputs,
                use_frozen=True,
            )
        )
        weight_integer, weight_scale, weight_zero = (
            self.weight_quantizer.quantize_to_int(self.classifier.weight)
        )
        centered_input = input_integer - input_zero
        centered_weight = weight_integer - weight_zero
        source_device = centered_input.device
        if source_device.type == "cpu":
            accumulator = centered_input @ centered_weight.transpose(0, 1)
        else:
            accumulator = centered_input.cpu() @ centered_weight.cpu().transpose(0, 1)
        if self.classifier.bias is not None:
            accumulator_scale = (input_scale * weight_scale).to(accumulator.device)
            integer_bias = torch.round(
                self.classifier.bias.detach().to(accumulator.device) / accumulator_scale
            ).to(torch.int32)
            accumulator = accumulator + integer_bias
        output_scale = (input_scale * weight_scale).to(accumulator.device)
        logits = (accumulator.float() * output_scale).to(source_device)
        return logits, accumulator

    def compute_loss(self, logits: Tensor, targets: Tensor) -> Tensor:
        return F.cross_entropy(logits, targets)

    def training_step(self, inputs: Tensor, targets: Tensor) -> Tensor:
        return self.compute_loss(self(inputs), targets)

### Task P2 · 학습·pruning·distillation update

`fit_portfolio_model`에서 optimizer, mask/EMA/quantization 경계와 update 순서를
드러냅니다. 비용 절감이 학습 경로에 미치는 영향을 함께 읽으세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §3: training with simulated quantization
- **이 셀의 책임:** Task 3
- **Tensor shape 계약:** float tensor [B,D] ↔ uint8/int32 tensor [B,D]와 dequantized [B,D]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** fake-quant loss/accuracy. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(model: IntegerQuantizationPortfolioModel) -> list[float]:
    inputs, targets = prepare_batch()
    model.calibrate_activations(inputs[:128])
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    for _ in range(model.config.steps):
        loss = model.training_step(inputs, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 품질과 비용을 함께 평가

accuracy 또는 reconstruction 품질과 parameter·sparsity·bit·latency를 별도로
기록합니다. 로컬 지표는 원문 하드웨어 benchmark의 대체값이 아닙니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §2.1, Eq. (1): r=S(q-Z) / §2.2, Eq. (2)–(4): quantized matmul / §2.4: zero-point and range / §3: training with simulated quantization
- **이 셀의 책임:** Task 1 / Task 2 / Task 1–2 / Task 3
- **Tensor shape 계약:** float tensor [B,D] ↔ uint8/int32 tensor [B,D]와 dequantized [B,D]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** scale/zero point와 round-trip / integer accumulator / train-only calibration scale 고정과 uint8 clamp / fake-quant loss/accuracy. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: IntegerQuantizationPortfolioModel,
) -> dict[str, float]:
    with torch.no_grad():
        fake_logits = model(features[test_idx])
        integer_logits, accumulator = model.integer_forward(features[test_idx])
        accuracy = (integer_logits.argmax(1) == labels[test_idx]).float().mean()
        inference_mse = F.mse_loss(integer_logits, fake_logits)
        calibration_inputs = features[train_idx[:128]]
        quantized = model.activation_quantizer(
            calibration_inputs,
            use_frozen=True,
        )
        roundtrip = F.mse_loss(quantized, calibration_inputs)
    return {
        "test_accuracy": float(accuracy.cpu()),
        "roundtrip_mse": float(roundtrip.cpu()),
        "integer_inference_mse": float(inference_mse.cpu()),
        "integer_accumulator_verified": float(accumulator.dtype == torch.int32),
        "integer_kernel_cpu_fallback": float(features.device.type != "cpu"),
        "accumulator_peak": float(accumulator.abs().max().cpu()),
        "activation_calibrated": float(
            model.activation_quantizer.calibrated.float().cpu()
        ),
        "activation_scale": float(model.activation_quantizer.frozen_scale.cpu()),
    }


portfolio_model = IntegerQuantizationPortfolioModel(PortfolioConfig())
portfolio_history = fit_portfolio_model(portfolio_model)
portfolio_metrics = evaluate_portfolio_model(portfolio_model)
assert np.isfinite(portfolio_history).all()
assert portfolio_metrics["activation_calibrated"] == 1.0
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 원 논문은 MobileNet/COCO와 실제 ARM integer kernel을 평가했다. 실습은 작은 linear layer의 수치 오차와 QAT proxy다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.